# 🧽 Extra — cleaning, further in

**Extra material.** Nothing in the assessed work assumes this notebook.

**Time:** ~30 minutes &nbsp;•&nbsp; **Goal:** three things that make the difference between a script that works and a script somebody else can use — readable names, a date index, and a file you can hand over.

## Run this first

In [ ]:
# Run this first.
import urllib.request
from pathlib import Path

import pandas as pd

DATA_URL = "https://raw.githubusercontent.com/HocheggerLab/y3-bio-python/main/lab06/data/"

SURVEY = "pollinators.csv"
if not Path(SURVEY).exists():
    urllib.request.urlretrieve(DATA_URL + SURVEY, SURVEY)

df = pd.read_csv(SURVEY)
df.columns = df.columns.str.strip()
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = pd.Categorical(
    df["Date"].dt.month_name(),
    categories=["April", "May", "June", "July", "August", "September"],
    ordered=True,
)

INSECTS = ["Beetles", "Hoverflies", "Flies", "Butterflies",
           "Honeybees", "Solitary_Bees", "Wasps", "Bumblebees"]

print(df.shape)
df.head(3)

## Skill 1 — Names that make code readable

`Crop_Generic`, `Total_Flowers`, `Solitary_Bees`. Somebody's spreadsheet conventions, now yours forever unless you change them.

```python
df = df.rename(columns={"Crop_Generic": "Crop", "Total_Flowers": "Flowers"})
```

And the values need the same treatment: `Rbean` and `Bbean` mean nothing to a reader, and will mean nothing to you in six months either.

```python
df["Crop"] = df["Crop"].replace({"Rbean": "Runner bean", "Bbean": "Broad bean"})
```

`.replace()` takes a dictionary of old → new and leaves anything not mentioned alone. `.map()` is its stricter sibling: it replaces everything, and turns anything missing from the dictionary into `NaN`. Use `.replace()` when you are patching a few values, `.map()` when you are translating a whole column and want to be told if you missed one.

In [ ]:
# DEMO - rename the columns, then translate the values
df.columns = df.columns.str.strip()
df = df.rename(columns={
    "Crop_Generic": "Crop",
    "Total_Flowers": "Flowers",
    "Solitary_Bees": "SolitaryBees",
})

df["Crop"] = df["Crop"].replace({"Rbean": "Runner bean", "Bbean": "Broad bean"})

print(df["Crop"].value_counts().head(8))
print()
print(df.columns.tolist())

### 🧹 DIY challenge

1. `Allotment` has a value `Moulsecoomb_Est` and another called `Moulsecoomb`. Look at how many watches each has. Are they one site or two? (You cannot tell from the data alone — say what you would need to ask.)
2. Underscores in site names are ugly in a figure legend. Replace them with spaces for the whole column using `df["Allotment"].str.replace("_", " ")`.
3. Use `.map()` with a dictionary covering only three crops, and look at what happens to the rest. Now you know why `.replace()` exists.

> 💡 `.str.` gives you Python's string methods, applied down a whole column — `.str.lower()`, `.str.strip()`, `.str.contains()`. It is the same `.str` that fixed the header in notebook 1.

In [ ]:
# TODO - your turn

## Skill 2 — Who did the surveying, and when?

Now that `Date` is a real date, `.dt` will answer questions the file never explicitly recorded.

In [ ]:
# DEMO - who did the surveying, and when?
print(df["Date"].dt.day_name().value_counts())

Seven watches on a Saturday out of 1,620. These are not weekend volunteers dropping by — this is organised fieldwork on working days. Worth knowing before you claim the survey represents "allotment visitors".

You could not have asked that question five minutes ago.

### 📅 DIY challenge

1. Add a column `DayOfYear` using `.dt.dayofyear`. What is its range?
2. How many watches happened in each **year**? Use `.dt.year`, and check it matches the existing `Year` column.
3. `(df["Date"] - df["Date"].min())` gives you a `Timedelta`. Print `.dt.days` on it to get a plain number of days since the survey started.

> 💡 A `.dt` method with brackets — `.dt.day_name()` — returns text. Without brackets — `.dt.month` — you get a number. Both are useful; they sort very differently.

In [ ]:
# TODO - your turn

## Skill 3 — An ordered column can be compared

An ordered `Categorical` does not just sort correctly. It also lets you ask comparison questions that would be nonsense on plain text:

```python
df[df["Month"] > "June"]    # late summer only
```

In [ ]:
# DEMO - comparisons that only work because the column is ordered
late = df[df["Month"] > "June"]
print(len(late), "watches in July, August and September")
print(late["Month"].value_counts().sort_index())

### 📅 DIY challenge

1. Sort the whole table by `Month` and print the first and last three rows' `Date`. Are they what you expect?
2. Build a small table of total `Bumblebees` per month, in **calendar order**, using `df.groupby("Month", observed=True)["Bumblebees"].sum()`.
3. Do the same for `Total_Flowers`. Now divide one by the other — bumblebee visits per flower, month by month. Which month is the busiest *per flower*, and is it the same as the month with the most bumblebees?

> 💡 Question 3 is the whole argument of session 3 in miniature. August has the most bumblebees because August has the most watches and the most flowers.

In [ ]:
# TODO - your turn

## Skill 4 — The index can be a date

So far the index has been row numbers, which carry no meaning. Put the dates there instead and the index starts doing work:

```python
ts = df.set_index("Date")
ts.loc["2018-07"]        # every watch in July 2018
ts.loc["2018-07-01":"2018-07-15"]   # a fortnight
```

pandas understands partial date strings. `"2018"`, `"2018-07"` and `"2018-07-12"` are all valid labels for `.loc`, and each selects the right slice. This is the moment `.loc` stops looking like a clumsy alternative to `.iloc` and starts looking like the point.

In [ ]:
# DEMO - a date index, and slicing by time
ts = df.set_index("Date").sort_index()

july18 = ts.loc["2018-07"]
print("watches in July 2018:", len(july18))
print("bumblebees seen:     ", july18["Bumblebees"].sum())

print()
fortnight = ts.loc["2018-08-01":"2018-08-15"]
print("first half of August 2018:", len(fortnight), "watches")

### 📅 DIY challenge — the shape of the season

Using the date-indexed table `ts`:

1. How many watches happened in **2017** versus **2018**? Do it with `.loc`, not with the `Year` column.
2. Pull out August 2017 and August 2018 separately. Compare the number of watches, the total bumblebees, and the total flowers.
3. Bumblebee visits **per flower** in August 2017 versus August 2018 — which year was better for bumblebees? Is your answer the same if you compare raw totals instead?

```python
```

> 💡 Question 3 will bite. 2018 has more than twice as many watches as 2017, so any raw total is mostly measuring how much surveying was done. This is the trap that session 3 is built around — you are meeting it early on purpose.

In [ ]:
# TODO - your turn

## Skill 5 — A clean table you can hand over

Cleaning is only finished when somebody else can pick the file up. That means every step is in the notebook, in order, so it can be re-run from the raw file — and the result is written out under a **new name**. Never overwrite your raw data.

In [ ]:
# DEMO - write it out, and prove it survives the round trip
df.to_csv("pollinators_clean.csv", index=False)

check = pd.read_csv("pollinators_clean.csv")
print("shape matches:", check.shape == df.shape)
print(check.columns.tolist()[:8])

## 🔭 Explore further

- Time series in pandas — the first two sections pay for themselves: <https://pandas.pydata.org/docs/user_guide/timeseries.html>
- `ts.resample("ME")["Bumblebees"].sum()` groups by calendar month automatically once the index is a date. Try `"W"` for weeks.
- Categorical data, including why it also saves a lot of memory: <https://pandas.pydata.org/docs/user_guide/categorical.html>

## Recap

You can now:

- rename columns and translate values with `.rename()`, `.replace()`, `.map()` and `.str`
- use `.dt` to ask questions the file never recorded, like which weekday a watch happened on
- compare and filter on an ordered `Categorical`
- put dates in the index and slice with partial date strings like `"2018-07"`
- write a clean file with `to_csv(..., index=False)`